In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

In [139]:
transactions = pd.read_csv("C:/Users/Admin/OneDrive - UWA/Projects/Data Engineering/Data-Engineering/Transaction Fraud Detection Project/Data/Raw/transactions.csv")
account_profiles = pd.read_csv("C:/Users/Admin/OneDrive - UWA/Projects/Data Engineering/Data-Engineering/Transaction Fraud Detection Project/Data/Raw/account_profiles.csv")
fraud_patterns = pd.read_csv("C:/Users/Admin/OneDrive - UWA/Projects/Data Engineering/Data-Engineering/Transaction Fraud Detection Project/Data/Raw/fraud_patterns.csv")
network_edges = pd.read_csv("C:/Users/Admin/OneDrive - UWA/Projects/Data Engineering/Data-Engineering/Transaction Fraud Detection Project/Data/Raw/network_edges.csv")
time_series_stats = pd.read_csv("C:/Users/Admin/OneDrive - UWA/Projects/Data Engineering/Data-Engineering/Transaction Fraud Detection Project/Data/Raw/time_series_stats.csv")  

Parse and convert *timestamp* column from string to datetime

In [140]:
transactions["timestamp"] = pd.to_datetime(transactions["timestamp"])

In [141]:
transactions["log_amount"] = np.log1p(transactions["amount"])

In [142]:
transactions["is_night"] = transactions["timestamp"].dt.hour.apply(lambda x: 1 if (x >= 0 and x < 5) else 0)

In [143]:
digital_channels = ["mobile_app", "web_browser", "phone_ivr"]
transactions.loc[~transactions["device_type"].isin(digital_channels), "ip_risk_score"] = 0

In [144]:
transactions["account_age_bucket"] = pd.cut(
    transactions["account_age_days"],
    bins = [-1, 90, 180, 365, float("inf")],
    labels = ["new_90d", "new_180d", "new_1y", "established"],
    right = True
)

age_order = {"new_90d": 0, "new_180d": 1, "new_1y": 2, "established": 3}
transactions["account_age_bucket_enc"] = transactions["account_age_bucket"].map(age_order)

In [145]:
transactions["is_unusual_spend"] = (transactions["amount_vs_avg_ratio"] > 5).astype(int)

In [146]:
transactions["is_high_velocity"] = (transactions["velocity_1h"] > 3).astype(int)

In [147]:
transactions["txn_to_limit_ratio"] = transactions["amount"] / transactions["credit_limit"]

In [148]:
account_dim = account_profiles[[
    "account_id",
    "home_country",
    "risk_score",
    "is_high_risk",
    "avg_monthly_txns",
    "pct_foreign",
    "avg_ip_risk",
    "unique_countries",
    "account_type"
]]

transactions = transactions.merge(account_dim, on = "account_id", how = "left")

Drop redundant columns *hour_of_day*, *day_of_week* as these data are all directly derivable from *timestamp* and *mcc_code* relates to *merchant_category*

Keeping these columns creates risks of inconsistency and bloats the feature matrix

In [149]:
transactions.drop(columns = ["day_of_week", "mcc_code", "account_age_bucket"], inplace = True)

One-hot encoding for Logistic regression

In [150]:
transactions_linear = transactions.copy()
transactions_linear = pd.get_dummies(transactions_linear, 
                                     columns = ["merchant_category", "device_type"],
                                     drop_first = True)
# merchant_country have too many unique values for one-hot encoding, 
# so we label encode them for logistic regression
le = LabelEncoder()
transactions_linear["merchant_country_enc"] = le.fit_transform(transactions_linear["merchant_country"].astype(str))

Label encoding for XGBoost and LightBGM notebook

In [151]:
for col in ["merchant_category", "merchant_country", "device_type"]:
    le = LabelEncoder()
    transactions[f"{col}_enc"] = le.fit_transform(transactions[col].astype(str))

In [159]:
# Columns to exclude from model input
EXCLUDE = [
    "transaction_id", "account_id",          # identifiers
    "timestamp",                              # datetime, features already extracted
    "merchant_category", "merchant_country",  # raw strings, encoded versions exist
    "device_type", "account_age_bucket",      # raw strings, encoded versions exist
    "account_type", "home_country",           # raw strings, encoded versions exist
    "fraud_pattern",                          # target-related, not a feature
    "is_fraud",                               # target variable
]

FEATURE_COLS = [c for c in transactions.columns if c not in EXCLUDE]
TARGET = "is_fraud"

print(f"Number of features (tree): {len(FEATURE_COLS)}")
print(f"Features (tree): {FEATURE_COLS}")

Number of features (tree): 28
Features (tree): ['hour_of_day', 'is_weekend', 'amount', 'card_present', 'device_known', 'ip_risk_score', 'is_foreign_txn', 'time_since_last_s', 'velocity_1h', 'amount_vs_avg_ratio', 'account_age_days', 'has_2fa', 'credit_limit', 'log_amount', 'is_night', 'account_age_bucket_enc', 'is_unusual_spend', 'is_high_velocity', 'txn_to_limit_ratio', 'risk_score', 'is_high_risk', 'avg_monthly_txns', 'pct_foreign', 'avg_ip_risk', 'unique_countries', 'merchant_category_enc', 'merchant_country_enc', 'device_type_enc']


In [153]:
EXCLUDE_LINEAR = [
    "transaction_id", "account_id",          # identifiers
    "timestamp",                              # datetime, features already extracted
    "merchant_country",                       # raw string, merchant_country_enc exists
    "account_type", "home_country",           # raw strings, not encoded
    "account_age_bucket",                     # already dropped
    "fraud_pattern",                          # target-related, not a feature
    "is_fraud",                               # target variable
    "amount",                                 # use log_amount for linear model
    "velocity_1h",                            # use is_high_velocity instead
    "amount_vs_avg_ratio",                    # use is_unusual_spend instead
]

FEATURE_COLS_LINEAR = [c for c in transactions_linear.columns if c not in EXCLUDE_LINEAR]
TARGET = "is_fraud"

print(f"Number of features (linear): {len(FEATURE_COLS_LINEAR)}")
print(f"Features (linear): {FEATURE_COLS_LINEAR}")

Number of features (linear): 40
Features (linear): ['hour_of_day', 'is_weekend', 'card_present', 'device_known', 'ip_risk_score', 'is_foreign_txn', 'time_since_last_s', 'account_age_days', 'has_2fa', 'credit_limit', 'log_amount', 'is_night', 'account_age_bucket_enc', 'is_unusual_spend', 'is_high_velocity', 'txn_to_limit_ratio', 'risk_score', 'is_high_risk', 'avg_monthly_txns', 'pct_foreign', 'avg_ip_risk', 'unique_countries', 'merchant_category_clothing', 'merchant_category_crypto', 'merchant_category_electronics', 'merchant_category_gambling', 'merchant_category_gas_station', 'merchant_category_grocery', 'merchant_category_hotel', 'merchant_category_money_transfer', 'merchant_category_online_retail', 'merchant_category_pharmacy', 'merchant_category_restaurant', 'merchant_category_travel', 'merchant_category_utilities', 'device_type_mobile_app', 'device_type_phone_ivr', 'device_type_pos_terminal', 'device_type_web_browser', 'merchant_country_enc']


In [158]:
#Tree models (XGBoost, LightGBM)
transactions.to_csv("C:/Users/Admin/OneDrive - UWA/Projects/Data Engineering/Data-Engineering/Transaction Fraud Detection Project/Data/Processed/transactions_tree.csv", index = False)

#Linear model (Logistic Regression)
transactions_linear.to_csv("C:/Users/Admin/OneDrive - UWA/Projects/Data Engineering/Data-Engineering/Transaction Fraud Detection Project/Data/Processed/transactions_linear.csv", index = False)